In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/05 20:41:47 WARN Utils: Your hostname, Ethans-Laptop.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.4 instead (on interface en0)
26/03/05 20:41:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/05 20:41:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/05 20:41:48 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [4]:
df_green = spark.read.option("recursiveFileLookup","true").parquet(
"/Users/ethandu/data-engineer-project/homework/HW6/data/pq/green"
)

In [5]:
df_green.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- lpep_pickup_datetime: timestamp (nullable = true)
 |-- lpep_dropoff_datetime: timestamp (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- trip_type: integer (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



In [6]:
df_green.rdd

MapPartitionsRDD[9] at javaToPython at NativeMethodAccessorImpl.java:0

In [8]:
```
SELECT  
    -- Reveneue grouping 
    PULocationID AS revenue_zone,
    date_trunc('hour',lpep_pickup_datetime) AS hour, 

    -- Revenue calculation 
  
    SUM(total_amount) AS amount,
   count(1) AS number_records

FROM
    green
where year(lpep_pickup_datetime) >='2020'
GROUP BY
    1, 2
order by 2,1
```

IndentationError: unindent does not match any outer indentation level (<string>, line 10)

In [22]:
rdd = df_green\
      .select ('lpep_pickup_datetime','PULocationID','total_amount')\
      .rdd

In [11]:
rdd.take(10)

[Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 29, 14, 32), PULocationID=97, total_amount=7.3),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 24, 4, 36, 44), PULocationID=7, total_amount=42.75),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 23, 13, 52), PULocationID=43, total_amount=8.3),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 26, 22, 18, 3), PULocationID=260, total_amount=6.8),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 4, 16, 0), PULocationID=177, total_amount=19.78),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 21, 23, 2, 4), PULocationID=24, total_amount=10.56),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 25, 19, 9, 59), PULocationID=82, total_amount=15.3),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 24, 21, 30, 37), PULocationID=260, total_amount=7.8),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 3, 11, 37, 21), PULocationID=130, total_amount=4.3),
 Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 9, 

In [12]:
rdd.filter(lambda row:True).take(1)

[Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 29, 14, 32), PULocationID=97, total_amount=7.3)]

In [13]:
from datetime import datetime

In [26]:
start = datetime(year =2020,month =1,day =1)

In [18]:
rdd.filter(lambda row : row.lpep_pickup_datetime >= start)

In [19]:
rows = rdd.take(10)
row = rows[0]

In [20]:
row

Row(lpep_pickup_datetime=datetime.datetime(2020, 1, 29, 14, 32), PULocationID=97, total_amount=7.3)

In [21]:
row.lpep_pickup_datetime.replace(minute =0, second =0,microsecond =0)

datetime.datetime(2020, 1, 29, 14, 0)

In [28]:
def prepare_for_grouping(row):
    hour = row.lpep_pickup_datetime.replace(minute =0, second =0,microsecond =0)
    zone = row.PULocationID
    key = (hour,zone)
    
    amount = row.total_amount
    count = 1
    value = (amount,count)

    return (key,value)

In [38]:
prepare_for_grouping(row)

((datetime.datetime(2020, 1, 29, 14, 0), 97), (7.3, 1))

Traceback (most recent call last):
  File "/Users/ethandu/data-engineer-project/homework/HW6/.venv/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
  File "/Users/ethandu/data-engineer-project/homework/HW6/.venv/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
    ~~~~~~~~~~~~~^^
BrokenPipeError: [Errno 32] Broken pipe


In [30]:
rdd.filter(lambda row : row.lpep_pickup_datetime >= start)\
   .map(prepare_for_grouping)\
   .take(10)

[((datetime.datetime(2020, 1, 29, 14, 0), 97), (7.3, 1)),
 ((datetime.datetime(2020, 1, 24, 4, 0), 7), (42.75, 1)),
 ((datetime.datetime(2020, 1, 23, 13, 0), 43), (8.3, 1)),
 ((datetime.datetime(2020, 1, 26, 22, 0), 260), (6.8, 1)),
 ((datetime.datetime(2020, 1, 4, 16, 0), 177), (19.78, 1)),
 ((datetime.datetime(2020, 1, 21, 23, 0), 24), (10.56, 1)),
 ((datetime.datetime(2020, 1, 25, 19, 0), 82), (15.3, 1)),
 ((datetime.datetime(2020, 1, 24, 21, 0), 260), (7.8, 1)),
 ((datetime.datetime(2020, 1, 3, 11, 0), 130), (4.3, 1)),
 ((datetime.datetime(2020, 1, 9, 7, 0), 218), (12.4, 1))]

In [31]:
def calculate_revenue (left_value,right_value): 
    left_amount ,left_count = left_value
    right_amount, right_count = right_value

    output_amount = left_amount +right_amount
    output_count = left_count +right_count

    return (output_amount,output_count)

In [37]:
rdd.filter(lambda row : row.lpep_pickup_datetime >= start)\
   .map(prepare_for_grouping)\
   .reduceByKey(calculate_revenue)\  # calculation of groupby 
   .toDF()\
   .take(10)

[Row(_1=Row(_1=datetime.datetime(2020, 1, 29, 14, 0), _2=97), _2=Row(_1=428.3700000000002, _2=33)),
 Row(_1=Row(_1=datetime.datetime(2020, 1, 26, 22, 0), _2=260), _2=Row(_1=114.32, _2=10)),
 Row(_1=Row(_1=datetime.datetime(2020, 1, 17, 18, 0), _2=181), _2=Row(_1=239.74000000000004, _2=16)),
 Row(_1=Row(_1=datetime.datetime(2020, 1, 14, 18, 0), _2=43), _2=Row(_1=363.4400000000001, _2=24)),
 Row(_1=Row(_1=datetime.datetime(2020, 1, 29, 10, 0), _2=89), _2=Row(_1=149.54999999999998, _2=5)),
 Row(_1=Row(_1=datetime.datetime(2020, 1, 9, 19, 0), _2=41), _2=Row(_1=632.8700000000001, _2=49)),
 Row(_1=Row(_1=datetime.datetime(2020, 1, 22, 18, 0), _2=43), _2=Row(_1=266.68000000000006, _2=17)),
 Row(_1=Row(_1=datetime.datetime(2020, 1, 30, 8, 0), _2=235), _2=Row(_1=168.54, _2=9)),
 Row(_1=Row(_1=datetime.datetime(2020, 1, 23, 19, 0), _2=66), _2=Row(_1=466.82000000000005, _2=28)),
 Row(_1=Row(_1=datetime.datetime(2020, 1, 29, 10, 0), _2=33), _2=Row(_1=571.4600000000003, _2=25))]

In [ ]:
def unwrap(row):
    return (row[0][0],row[0][1], row[1][0],row[1][1])
    